# Parkinson's Disease Severity Prediction
**Dataset:** Parkinson's Telemonitoring — UCI ML Repository (ID: 189)  
**Source:** Biomedical voice measurements from 42 patients with early-stage Parkinson's disease

## Step 1: Import Dataset

In [1]:
import pandas as pd

df = pd.read_json('parkinsons_updrs.json')

target_cols  = ['motor_UPDRS', 'total_UPDRS']
feature_cols = [c for c in df.columns if c not in target_cols + ['subject#']]

X = df[feature_cols]
y = df[target_cols]

print(f'Features : {X.shape}')
print(f'Targets  : {y.shape}')
print(f'Columns  : {feature_cols}')


Features : (5875, 19)
Targets  : (5875, 2)
Columns  : ['age', 'sex', 'test_time', 'Jitter(%)', 'Jitter(Abs)', 'Jitter:RAP', 'Jitter:PPQ5', 'Jitter:DDP', 'Shimmer', 'Shimmer(dB)', 'Shimmer:APQ3', 'Shimmer:APQ5', 'Shimmer:APQ11', 'Shimmer:DDA', 'NHR', 'HNR', 'RPDE', 'DFA', 'PPE']


## Step 2: Problem Definition & Goal

**Problem Type:** Supervised Regression

**Goal:** Predict the `motor_UPDRS` score — the Unified Parkinson's Disease Rating Scale (motor subscale) — from 16 biomedical voice measurements recorded remotely.  
A lower UPDRS score indicates better motor function; a higher score indicates greater symptom severity.

**Why this matters:** Accurate remote prediction of UPDRS enables clinicians to monitor disease progression without in-person visits, improving accessibility and enabling timely intervention.

**Features (16 voice metrics):** Jitter, Shimmer, NHR, HNR, RPDE, DFA, PPE, and others.  
**Target:** `motor_UPDRS` (continuous, range 0–108)

In [2]:
import numpy as np

y_target = y['motor_UPDRS']

print("=== Dataset Overview ===")
print(f"Features shape : {X.shape}")
print(f"Target shape   : {y_target.shape}")
print(f"\nFeatures       : {X.columns.tolist()}")
print(f"\nMissing values : {X.isnull().sum().sum()}")
print("\n=== Target Statistics ===")
print(y_target.describe())
print("\n=== First 5 rows ===")
display(X.head())


=== Dataset Overview ===
Features shape : (5875, 19)
Target shape   : (5875,)

Features       : ['age', 'sex', 'test_time', 'Jitter(%)', 'Jitter(Abs)', 'Jitter:RAP', 'Jitter:PPQ5', 'Jitter:DDP', 'Shimmer', 'Shimmer(dB)', 'Shimmer:APQ3', 'Shimmer:APQ5', 'Shimmer:APQ11', 'Shimmer:DDA', 'NHR', 'HNR', 'RPDE', 'DFA', 'PPE']

Missing values : 0

=== Target Statistics ===
count    5875.000000
mean       21.296229
std         8.129282
min         5.037700
25%        15.000000
50%        20.871000
75%        27.596500
max        39.511000
Name: motor_UPDRS, dtype: float64

=== First 5 rows ===


,age,sex,test_time,Jitter(%),Jitter(Abs),Jitter:RAP,Jitter:PPQ5,Jitter:DDP,Shimmer,Shimmer(dB),Shimmer:APQ3,Shimmer:APQ5,Shimmer:APQ11,Shimmer:DDA,NHR,HNR,RPDE,DFA,PPE
0,72,0,5.6431,0.00662,0.000034,0.00401,0.00317,0.01204,0.02565,0.230,0.01438,0.01309,0.01662,0.04314,0.014290,21.640,0.41888,0.54842,0.16006
1,72,0,12.6660,0.00300,0.000017,0.00132,0.00150,0.00395,0.02024,0.179,0.00994,0.01072,0.01689,0.02982,0.011112,27.183,0.43493,0.56477,0.10810
2,72,0,19.6810,0.00481,0.000025,0.00205,0.00208,0.00616,0.01675,0.181,0.00734,0.00844,0.01458,0.02202,0.020220,23.047,0.46222,0.54405,0.21014
3,72,0,25.6470,0.00528,0.000027,0.00191,0.00264,0.00573,0.02309,0.327,0.01106,0.01265,0.01963,0.03317,0.027837,24.445,0.48730,0.57794,0.33277
4,72,0,33.6420,0.00335,0.000020,0.00093,0.00130,0.00278,0.01703,0.176,0.00679,0.00929,0.01819,0.02036,0.011625,26.126,0.47188,0.56122,0.19361


## Step 3: Model Selection

Three models are selected, ranging from simple to complex:

| Model | Reason |
|---|---|
| **Linear Regression** | Baseline — fast, interpretable, assumes linear relationships |
| **Random Forest Regressor** | Ensemble of trees — handles non-linearity, robust to outliers |
| **Gradient Boosting Regressor** | Sequential boosting — typically highest accuracy on tabular data |

## Step 4: Training the Models

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# Train/test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_target, test_size=0.2, random_state=42
)

# Scale features for Linear Regression
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# --- Linear Regression ---
lr = LinearRegression()
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)

# --- Random Forest ---
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

# --- Gradient Boosting ---
gb = GradientBoostingRegressor(n_estimators=100, random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)

# Compare baseline results
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest', 'Gradient Boosting'],
    'RMSE':  [np.sqrt(mean_squared_error(y_test, y_pred_lr)),
              np.sqrt(mean_squared_error(y_test, y_pred_rf)),
              np.sqrt(mean_squared_error(y_test, y_pred_gb))],
    'R²':    [r2_score(y_test, y_pred_lr),
              r2_score(y_test, y_pred_rf),
              r2_score(y_test, y_pred_gb)]
})
results = results.round(4)
print("=== Baseline Model Comparison ===")
display(results)


=== Baseline Model Comparison ===


,Model,RMSE,R²
0,Linear Regression,7.4843,0.1224
1,Random Forest,1.3051,0.9733
2,Gradient Boosting,3.9586,0.7545


## Step 5: Parameter Selection (Hyperparameter Tuning)

We apply **GridSearchCV** with 5-fold cross-validation on the Random Forest model to search for the best combination of:
- `n_estimators` — number of trees
- `max_depth` — maximum tree depth
- `min_samples_split` — minimum samples required to split a node

In [4]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators':     [50, 100, 200],
    'max_depth':        [None, 5, 10],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train)

best_rf   = grid_search.best_estimator_
y_pred_best = best_rf.predict(X_test)

print(f"Best parameters : {grid_search.best_params_}")
print(f"Tuned RF  RMSE  : {np.sqrt(mean_squared_error(y_test, y_pred_best)):.4f}")
print(f"Tuned RF  R²    : {r2_score(y_test, y_pred_best):.4f}")


Fitting 5 folds for each of 27 candidates, totalling 135 fits
Best parameters : {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}
Tuned RF  RMSE  : 1.3045
Tuned RF  R²    : 0.9733


## Step 6: Visualizations

In [5]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# ── Visualization 1: Feature Correlation Heatmap ──────────────────────────────
corr = X.corr().round(2)

fig1 = px.imshow(
    corr,
    text_auto=True,
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    aspect='auto',
    title='<b>Visualization 1: Feature Correlation Heatmap</b>'
)
fig1.update_layout(width=900, height=700)
fig1.show()


In [6]:
# ── Visualization 2: Distribution of Target Variable ─────────────────────────
fig2 = px.histogram(
    x=y_target,
    nbins=50,
    marginal='box',
    title='<b>Visualization 2: Distribution of motor_UPDRS</b>',
    labels={'x': 'motor_UPDRS Score'},
    color_discrete_sequence=['steelblue']
)
fig2.update_layout(xaxis_title='motor_UPDRS Score', yaxis_title='Count')
fig2.show()


In [7]:
# ── Visualization 3: Actual vs. Predicted (All Models) ───────────────────────
perfect_line = [y_test.min(), y_test.max()]

fig3 = go.Figure()
fig3.add_trace(go.Scatter(
    x=y_test, y=y_pred_lr,
    mode='markers', name='Linear Regression',
    marker=dict(opacity=0.6, size=5)
))
fig3.add_trace(go.Scatter(
    x=y_test, y=y_pred_rf,
    mode='markers', name='Random Forest (default)',
    marker=dict(opacity=0.6, size=5)
))
fig3.add_trace(go.Scatter(
    x=y_test, y=y_pred_best,
    mode='markers', name='Random Forest (tuned)',
    marker=dict(opacity=0.6, size=5)
))
fig3.add_trace(go.Scatter(
    x=perfect_line, y=perfect_line,
    mode='lines', name='Perfect Prediction',
    line=dict(dash='dash', color='black', width=2)
))
fig3.update_layout(
    title='<b>Visualization 3: Actual vs. Predicted motor_UPDRS</b>',
    xaxis_title='Actual motor_UPDRS',
    yaxis_title='Predicted motor_UPDRS',
    width=800, height=550
)
fig3.show()


In [8]:
# ── Visualization 4: Feature Importance (Tuned Random Forest) ────────────────
feat_imp = (
    pd.DataFrame({'Feature': X.columns, 'Importance': best_rf.feature_importances_})
    .sort_values('Importance', ascending=True)
)

fig4 = px.bar(
    feat_imp,
    x='Importance', y='Feature',
    orientation='h',
    title='<b>Visualization 4: Feature Importance — Tuned Random Forest</b>',
    color='Importance',
    color_continuous_scale='Turbo'
)
fig4.update_layout(height=500, yaxis_title='', coloraxis_showscale=False)
fig4.show()


In [9]:
# ── Visualization 5: Model Comparison Bar Chart (RMSE & R²) ──────────────────
results_full = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest (default)', 'Gradient Boosting', 'Random Forest (tuned)'],
    'RMSE':  [
        np.sqrt(mean_squared_error(y_test, y_pred_lr)),
        np.sqrt(mean_squared_error(y_test, y_pred_rf)),
        np.sqrt(mean_squared_error(y_test, y_pred_gb)),
        np.sqrt(mean_squared_error(y_test, y_pred_best))
    ],
    'R²': [
        r2_score(y_test, y_pred_lr),
        r2_score(y_test, y_pred_rf),
        r2_score(y_test, y_pred_gb),
        r2_score(y_test, y_pred_best)
    ]
})

fig5 = make_subplots(
    rows=1, cols=2,
    subplot_titles=['RMSE (lower is better)', 'R² Score (higher is better)']
)
fig5.add_trace(
    go.Bar(x=results_full['Model'], y=results_full['RMSE'],
           marker_color='tomato', name='RMSE'),
    row=1, col=1
)
fig5.add_trace(
    go.Bar(x=results_full['Model'], y=results_full['R²'],
           marker_color='mediumseagreen', name='R²'),
    row=1, col=2
)
fig5.update_layout(
    title_text='<b>Visualization 5: Model Performance Comparison</b>',
    showlegend=False,
    width=1000, height=450
)
fig5.update_xaxes(tickangle=15)
fig5.show()
